# 🌽 rs-embed — From Satellites to Crop Yield in One Pipeline
### *Illinois maize, all in one notebook* — SIGSPATIAL '26, UC Riverside

A single, linear story that walks from a raw location to a trained yield model —
the "all-in-one" companion to the interactive scenarios notebook.

1. **Act 1 — the one-line reveal.** Turn one Illinois field + a summer window
   into an embedding with a single call.
2. **Act 2 — swap the model.** Change *one string* to see how four different
   foundation models "see" the same field.
3. **Act 3 — the payoff.** Embed many fields across all models on **shared
   imagery**, then train a yield regressor per model and rank them — fairly.

> Runs **cached & offline by default** (`RUN_LIVE = False`). Flip it to `True`
> with Earth Engine auth + a GPU to compute live; failures fall back to cache.


## Act 0 — the pain we're replacing

To predict maize yield from space you normally first fight the imagery: Earth
Engine auth, cloud-free compositing, the right bands per model, each model's
normalization, checkpoints, forward passes. *Then* you get to the actual
science. Below, that entire first stage is **one function call** — so the
notebook is mostly about the yield model, the way it should be.


In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────
import sys, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

HERE = Path.cwd()
for cand in [HERE, HERE / "examples", *HERE.parents]:
    if (cand / "iguide_demo_helpers.py").exists():
        sys.path.insert(0, str(cand)); break
import iguide_demo_helpers as H
from build_demo_cache import MODEL_LIST, PRECOMPUTED_MODEL, MAIZE_TEMPORAL, MAIZE_YEAR, BUFFER_M

# ⚙️  Flip to True with GEE auth + GPU to run the models live.
RUN_LIVE = False

cache = H.DemoCache.find(HERE)
print("Cache dir :", cache.root)
print("Cache ready:", cache.available, "| RUN_LIVE:", RUN_LIVE)
if not cache.available:
    print("\n⚠️  No cache yet. Build it once with:\n"
          "      python examples/build_demo_cache.py --only maize"
          "   # needs `earthengine authenticate` + SPAM CSVs\n"
          "    Each act below shows a friendly notice until then.")

RS = None
if RUN_LIVE:
    try:
        import ee
        try:
            ee.Initialize()
        except Exception:
            ee.Authenticate(); ee.Initialize()
        from rs_embed import OutputSpec, PointBuffer, TemporalSpec, get_embedding
        RS = dict(get_embedding=get_embedding, PointBuffer=PointBuffer,
                  TemporalSpec=TemporalSpec, OutputSpec=OutputSpec)
        print("Live rs-embed ready ✔")
    except Exception as e:
        print("Live mode unavailable → using cache. Reason:", repr(e)); RUN_LIVE = False

# The single Illinois maize field used for the Act 1–2 'reveal'.
_lab = cache.maize_labels()
REVEAL = (float(_lab["lon"][0]), float(_lab["lat"][0])) if _lab is not None else (-88.9, 40.1)
print("Reveal field (lon,lat):", REVEAL)

## Act 1 — the one-line reveal

One call turns *where + when + which model* into an embedding. No band lists, no
normalization, no checkpoints in sight.

```python
emb = get_embedding("satmae", spatial=PointBuffer(lon, lat, 1280),
                    temporal=TemporalSpec.range("2019-06-01", "2019-08-31"),
                    output=OutputSpec.grid())
```


In [ ]:
scene = cache.maize_scene()
model1 = MODEL_LIST[0]

def reveal_grid(model):
    if RUN_LIVE and RS is not None:
        try:
            temporal = (RS["TemporalSpec"].year(MAIZE_YEAR) if model == PRECOMPUTED_MODEL
                        else RS["TemporalSpec"].range(*MAIZE_TEMPORAL))
            emb = RS["get_embedding"](model,
                spatial=RS["PointBuffer"](lon=REVEAL[0], lat=REVEAL[1], buffer_m=BUFFER_M),
                temporal=temporal, output=RS["OutputSpec"].grid())
            return H.to_dhw(emb.data), "live"
        except Exception as e:
            print("live embed failed → cache:", repr(e))
    if scene is not None and f"grid__{model}" in scene:
        return H.to_dhw(scene[f"grid__{model}"]), "cached"
    return None, "missing"

if scene is None and not RUN_LIVE:
    print("⚠️  maize/scene.npz missing — run: python build_demo_cache.py --only maize")
else:
    grid, how = reveal_grid(model1)
    if grid is None:
        print(f"No grid for {model1}.")
    else:
        print(f"'{model1}' embedding for the Illinois field [{how}] → grid shape {grid.shape} "
              f"(D={grid.shape[0]} feature channels)")
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(H.pca_rgb(grid)); ax[0].set_title(f"{model1}: embedding (PCA-RGB)"); ax[0].axis("off")
        vec = H.pooled_vector(grid)
        ax[1].plot(vec, lw=0.8, color="#2e7d32"); ax[1].set_title(f"pooled vector ({vec.shape[0]}-d)")
        ax[1].set_xlabel("dimension"); plt.tight_layout(); plt.show()

## Act 2 — swap the model

The whole point of rs-embed: change **one string** and a completely different
foundation model processes the *same* field. Here are four, side by side.


In [ ]:
models = [m for m in MODEL_LIST if (scene is not None and f"grid__{m}" in scene)] or MODEL_LIST
if scene is None and not RUN_LIVE:
    print("⚠️  maize/scene.npz missing — run: python build_demo_cache.py --only maize")
else:
    grids = [(m, reveal_grid(m)[0]) for m in models]
    grids = [(m, g) for m, g in grids if g is not None]
    n = len(grids)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    if n == 1: axes = [axes]
    for ax, (m, g) in zip(axes, grids):
        ax.imshow(H.pca_rgb(g)); ax.set_title(m, fontsize=11); ax.axis("off")
    fig.suptitle("Same Illinois field · same code · different foundation model", y=1.02)
    plt.tight_layout(); plt.show()
    print("Each panel was produced by changing only the model name string.")

## Act 3 — the payoff: which model predicts yield best?

Now scale up. `export_batch` embeds **many** Illinois fields across **all**
models — fetching each field's imagery **once and sharing it** (`dedup_reused`)
— then we train a RandomForest yield regressor per model on the **same**
train/test split. Only the model differs, so the ranking is fair.


In [ ]:
mdir = cache.maize_dir()
labels = cache.maize_labels()
bundle = mdir / "maize_export.npz"
if labels is None or not bundle.exists():
    print("⚠️  maize bundle/labels missing — run: python build_demo_cache.py --only maize")
else:
    feats, manifest = H.load_export_features(bundle)
    y = np.asarray(labels["y"]).ravel()
    board = H.regression_leaderboard(feats, y, test_size=0.3, seed=42)
    if not board:
        print("No models aligned with labels — re-run the maize build.")
    else:
        print("Identical fields, identical split — only the model differs.\n")
        print(f"{'rank':<5}{'model':<14}{'R^2':>8}{'MAE':>8}{'RMSE':>8}{'dim':>7}")
        for r, s in enumerate(board, 1):
            print(f"{r:<5}{s.model:<14}{s.r2:>8.3f}{s.mae:>8.2f}{s.rmse:>8.2f}{s.dim:>7}")

        best = board[0]
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        names = [s.model for s in board]; r2s = [s.r2 for s in board]
        ax[0].barh(names[::-1], r2s[::-1], color="#2e7d32")
        for i, v in enumerate(r2s[::-1]):
            ax[0].text(v, i, f" {v:.3f}", va="center", fontsize=8)
        ax[0].set_xlabel("test R²"); ax[0].set_title("🏆 Yield-prediction leaderboard")

        # true-vs-pred scatter for the winning model (same split, refit for plot)
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.model_selection import train_test_split
        X = np.nan_to_num(np.asarray(feats[best.model], dtype="f4"))
        itr, ite = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42)
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X[itr], y[itr])
        pred = rf.predict(X[ite])
        ax[1].scatter(y[ite], pred, s=22, alpha=.7, color="#1565c0")
        lim = [float(min(y[ite].min(), pred.min())), float(max(y[ite].max(), pred.max()))]
        ax[1].plot(lim, lim, "--", color="#999")
        ax[1].set_xlabel("true yield"); ax[1].set_ylabel("predicted yield")
        ax[1].set_title(f"Best model: {best.model}  (R²={best.r2:.3f})")
        plt.tight_layout(); plt.show()

        reused = [m.get("inputs", {}).get("dedup_reused")
                  for m in manifest.get("models", []) if isinstance(m.get("inputs"), dict)]
        print(f"\nFairness — imagery fetched once & shared across models "
              f"(dedup_reused observed: {bool(any(reused))}).")

## Takeaways

- **One pipeline, satellites → yield.** The hard 90% (imagery + per-model
  preprocessing) was one call per model; the science (the regressor) was the easy part.
- **Models are interchangeable.** Acts 1–2 swapped foundation models by editing a
  single string — no glue, no silent band-order bugs.
- **The comparison is fair by construction.** Act 3 ran on a shared, audited
  `export_batch` manifest, so the leaderboard reflects the model, not the plumbing.

**See also:** `examples/iguide_demo.ipynb` (the interactive *Earth's Twins /
Time Machine / Land-Cover / Showdown* console) and `examples/iguide_demo.html`.
Run live by setting `RUN_LIVE = True` after `earthengine authenticate`.
